# 04 Comparison and Condition Data

This notebook combines the reference, RADAR, and LiDAR velocity results and prepares the condition data used by the later condition-based analyses.

The original comparison and condition-data logic from `condition_velocity.ipynb` is retained.

The final comparison and condition tables are saved to `df/condition/`.

In [1]:
import sys
sys.path.append("../..")

from pathlib import Path

import pandas as pd
from truckscenes import TruckScenes

from src.config import DATA_ROOT
from src.comparison import build_velocity_comparison_df
from src.condition.condition_config import CONDITIONS
from src.condition.condition_data import (
    build_condition_table,
    build_condition_inventory,
)

In [ ]:
RESULT_DIR = Path("../../df/condition")

## 1. Load Previous Results

In [3]:
keyframe_df = pd.read_csv(RESULT_DIR / "keyframe_df.csv")
vehicle_reference_df = pd.read_csv(RESULT_DIR / "vehicle_reference_df.csv")
radar_velocity_df = pd.read_csv(RESULT_DIR / "radar_velocity_df.csv")
lidar_velocity_df = pd.read_csv(RESULT_DIR / "lidar_velocity_df.csv")
trucksc = TruckScenes(dataroot=DATA_ROOT)

Auto-detected latest version: v1.2-trainval
Loading truckscenes tables for version v1.2-trainval...
11 attribute,
18 calibrated_sensor,
27 category,
1200243 ego_motion_cabin,
1200183 ego_motion_chassis,
1201993 ego_pose,
43769 instance,
23902 sample,
819536 sample_annotation,
2586264 sample_data,
598 scene,
18 sensor,
4 visibility,
598 weather_annotation,
Done loading in 55.837 seconds.
Reverse indexing ...
Done reverse indexing in 37.3 seconds.
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 2. Build Comparison Data

In [4]:
local_sample_tokens = set(keyframe_df[keyframe_df["Modality"].isin(["RADAR", "LiDAR"])]["Sample Token"])

local_vehicle_reference_df = vehicle_reference_df[
    vehicle_reference_df["start_sample_token"].isin(local_sample_tokens)
    & vehicle_reference_df["end_sample_token"].isin(local_sample_tokens)
].copy()

In [5]:
comparison_df = build_velocity_comparison_df(
    local_vehicle_reference_df,
    radar_velocity_df,
    lidar_velocity_df,
)

interval_key = ["start_annotation_token", "end_annotation_token"]

comparison_df = comparison_df.merge(
    radar_velocity_df[interval_key + ["is_stable_2d"]],
    on=interval_key,
    how="left",
)

comparison_df["radar_reconstructable_available"] = comparison_df["radar_velocity_available"]
comparison_df["radar_stable_available"] = comparison_df["is_stable_2d"].fillna(False)
comparison_df["radar_velocity_available"] = comparison_df["radar_stable_available"]

In [6]:
print("Reference intervals:", len(comparison_df))
print("RADAR reconstructable:", comparison_df["radar_reconstructable_available"].sum())
print("RADAR stable:", comparison_df["radar_stable_available"].sum())
print("LiDAR available:", comparison_df["lidar_velocity_available"].sum())

Reference intervals: 206040
RADAR reconstructable: 97517
RADAR stable: 38179
LiDAR available: 96551


In [7]:
comparison_df.to_csv(RESULT_DIR / "comparison_df.csv", index=False)
print("Comparison data saved.")

Comparison data saved.


## 3. Build Condition Data

In [8]:
local_sample_tokens = set(keyframe_df[keyframe_df["Modality"].isin(["RADAR", "LiDAR"])]["Sample Token"])

In [9]:
condition_df = build_condition_table(trucksc, sample_tokens=local_sample_tokens)
condition_inventory = build_condition_inventory(condition_df, CONDITIONS)
condition_inventory

,condition,value,scenes,samples,objects,annotations
0,area,city,18,720,2106,49888
1,area,highway,180,7188,12368,215722
2,area,other_area,1,40,56,1362
3,area,parking,2,80,107,3511
4,area,residential,6,240,503,8864
5,area,rural,23,915,1380,27110
6,area,terminal,22,880,460,13606
7,weather,clear,145,5793,9591,169812
8,weather,fog,10,395,588,8293
9,weather,other_weather,1,40,24,754


## 4. Save Condition Data

In [10]:
condition_df.to_csv(RESULT_DIR / "condition_df.csv", index=False)
print("Condition data saved.")

Condition data saved.


## 5. Setup Summary

In [11]:
setup_summary = pd.DataFrame({
    "dataset": [
        "Reference intervals",
        "RADAR velocity intervals",
        "LiDAR velocity intervals",
        "Comparison intervals",
        "Condition annotations",
    ],
    "rows": [
        len(local_vehicle_reference_df),
        len(radar_velocity_df),
        len(lidar_velocity_df),
        len(comparison_df),
        len(condition_df),
    ],
})

setup_summary

,dataset,rows
0,Reference intervals,206040
1,RADAR velocity intervals,97517
2,LiDAR velocity intervals,96551
3,Comparison intervals,206040
4,Condition annotations,320063
